In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [2]:
import os

# Get the folder where this notebook is running from
notebook_dir = os.getcwd()

# Go one level up to reach the project root (Summer_Projects folder)
project_root = os.path.dirname(notebook_dir)

# Path to the raw file — we READ from here, NEVER write to it
raw_path = os.path.join(project_root, 'data', 'raw', 'inventory_raw.xlsx')

# Paths where the cleaned data will be saved
cleaned_xlsx = os.path.join(project_root, 'data', 'cleaned', 'inventory_cleaned.xlsx')
cleaned_csv  = os.path.join(project_root, 'data', 'cleaned', 'inventory_cleaned.csv')

# Confirm the paths look correct
print("Notebook folder :", notebook_dir)
print("Project root    :", project_root)
print("Raw data path   :", raw_path)
print("Cleaned xlsx    :", cleaned_xlsx)
print("Cleaned csv     :", cleaned_csv)

# Verify the raw file actually exists before we try to open it
if os.path.exists(raw_path):
    print("\nRaw file found. Ready to load.")
else:
    print("\nERROR: Raw file not found. Check the path above.")


Notebook folder : c:\Users\DELL\Documents\Summer_Projects\notebook
Project root    : c:\Users\DELL\Documents\Summer_Projects
Raw data path   : c:\Users\DELL\Documents\Summer_Projects\data\raw\inventory_raw.xlsx
Cleaned xlsx    : c:\Users\DELL\Documents\Summer_Projects\data\cleaned\inventory_cleaned.xlsx
Cleaned csv     : c:\Users\DELL\Documents\Summer_Projects\data\cleaned\inventory_cleaned.csv

Raw file found. Ready to load.


In [3]:
df = pd.read_excel(raw_path)

print(f"Loaded successfully.")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")


Loaded successfully.
Shape: 201 rows x 8 columns


In [4]:
df.head()


,Product_ID,Product_Name,Category,Specification,Quantity,Sell_Price_ETB,Location,Count_Date
0,P0001,Abc Panel,Panel Light,12W,4.0,400,Shop,46192
1,P0002,Bright Panel,Panel Light,36W,23.0,600,Shop,46192
2,P0003,Diamond Burried,Panel Light,15W,1.0,500,Shop,46192
3,P0004,Glory Normal Panel,Panel Light,18W,3.0,450,Shop,46192
4,P0005,Glory Normal Panel,Panel Light,24W,19.0,550,Shop,46192


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product_ID      201 non-null    str    
 1   Product_Name    201 non-null    str    
 2   Category        201 non-null    str    
 3   Specification   152 non-null    str    
 4   Quantity        191 non-null    float64
 5   Sell_Price_ETB  201 non-null    int64  
 6   Location        201 non-null    str    
 7   Count_Date      201 non-null    int64  
dtypes: float64(1), int64(2), str(5)
memory usage: 19.0 KB


In [6]:
df.describe()


,Quantity,Sell_Price_ETB,Count_Date
count,191.000000,201.000000,201.0
mean,13.350785,826.069652,46192.0
std,22.013094,952.493819,0.0
min,0.000000,80.000000,46192.0
25%,2.500000,220.000000,46192.0
50%,5.000000,375.000000,46192.0
75%,17.000000,1200.000000,46192.0
max,200.000000,4800.000000,46192.0


In [7]:
df_clean = df.copy()

print(f"Working copy created.")
print(f"Original df    : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Copy df_clean  : {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")


Working copy created.
Original df    : 201 rows x 8 columns
Copy df_clean  : 201 rows x 8 columns


In [8]:
df_clean.rename(columns={'Sell_Price_ETB': 'Purchase_Price_ETB'}, inplace=True)

print("Column renamed successfully.")
print("Columns now:", list(df_clean.columns))


Column renamed successfully.
Columns now: ['Product_ID', 'Product_Name', 'Category', 'Specification', 'Quantity', 'Purchase_Price_ETB', 'Location', 'Count_Date']


In [9]:
# Check what the column looks like right now
print("Before conversion:")
print("  Data type  :", df_clean['Count_Date'].dtype)
print("  First value:", df_clean['Count_Date'].iloc[0])
print()

# Convert the Excel serial number to a real date
# unit='D' means the number represents days
# origin='1899-12-30' is Excel's starting point (Excel's own epoch)
df_clean['Count_Date'] = pd.to_datetime(
    df_clean['Count_Date'],
    unit='D',
    origin='1899-12-30',
    errors='coerce'
)

# Check the result
print("After conversion:")
print("  Data type  :", df_clean['Count_Date'].dtype)
print("  First value:", df_clean['Count_Date'].iloc[0].date())
print("  All unique dates:", df_clean['Count_Date'].dt.date.unique())


Before conversion:
  Data type  : int64
  First value: 46192

After conversion:
  Data type  : datetime64[s]
  First value: 2026-06-19
  All unique dates: [datetime.date(2026, 6, 19)]


In [10]:
# Count missing values in each column
missing_count = df_clean.isnull().sum()

# Calculate what percentage of each column is missing
missing_percent = (df_clean.isnull().sum() / len(df_clean) * 100).round(1)

# Combine into one neat summary table
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percent
})

# Only show columns that actually have missing values
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

print("Columns with missing values:")
print()
print(missing_summary)
print()
print(f"Total cells in dataset : {df_clean.size}")
print(f"Total missing cells    : {df_clean.isnull().sum().sum()}")


Columns with missing values:

               Missing Count  Missing %
Specification             49       24.4
Quantity                  10        5.0

Total cells in dataset : 1608
Total missing cells    : 59


In [11]:
# --- Fix 1: Specification ---
# Fill missing specs with 'N/A' — means Not Applicable, not unknown
df_clean['Specification'] = df_clean['Specification'].fillna('N/A')

print("Specification — missing values remaining:",
      df_clean['Specification'].isnull().sum())

# --- Fix 2: Quantity ---
# First, show WHICH products have missing quantity
# so we know exactly what needs a recount
missing_qty_products = df_clean[df_clean['Quantity'].isnull()]
print()
print("Products with missing Quantity (need recount):")
print()
print(missing_qty_products[['Product_ID', 'Product_Name',
                             'Category', 'Purchase_Price_ETB']].to_string(index=False))

# Now fill missing Quantity with 0
df_clean['Quantity'] = df_clean['Quantity'].fillna(0)

print()
print("Quantity — missing values remaining:",
      df_clean['Quantity'].isnull().sum())


Specification — missing values remaining: 0

Products with missing Quantity (need recount):

Product_ID          Product_Name       Category  Purchase_Price_ETB
     P0071            Usipu Warm           Lamp                 120
     P0135        Magnetic Light Magnetic Light                2700
     P0136        Magnetic Light Magnetic Light                3200
     P0137             Phase Bar Magnetic Light                1500
     P0138          Power Supply Magnetic Light                2500
     P0141              Zero LED Magnetic Light                2500
     P0142              Zero LED Magnetic Light                3000
     P0193 Single Aluminium Warm    Strip Light                 165
     P0196    Strip Socket 3 Pin    Strip Light                 300
     P0200     Warm + Blue Strip    Strip Light                 125

Quantity — missing values remaining: 0


In [12]:
# Check the type before converting
print("Before:", df_clean['Quantity'].dtype)

# Convert Quantity to integer — no decimals needed for product counts
df_clean['Quantity'] = df_clean['Quantity'].astype(int)

# Check the type after converting
print("After :", df_clean['Quantity'].dtype)
print()

# Verify with a sample — quantity values should now show without .0
print("Sample quantities:")
print(df_clean[['Product_ID', 'Product_Name', 'Quantity']].head(5).to_string(index=False))


Before: float64
After : int64

Sample quantities:
Product_ID       Product_Name  Quantity
     P0001          Abc Panel         4
     P0002       Bright Panel        23
     P0003    Diamond Burried         1
     P0004 Glory Normal Panel         3
     P0005 Glory Normal Panel        19


In [13]:
# Define which columns contain text that needs standardizing
# We do NOT include Specification — specs like '10A', '36W', 'RGB'
# should not be title-cased (e.g. '10a' would be wrong)
text_columns = ['Product_Name', 'Category', 'Location']

for col in text_columns:
    # Step 1: Remove spaces from both ends
    # '  Tesla Panel BK  ' becomes 'Tesla Panel BK'
    df_clean[col] = df_clean[col].str.strip()

    # Step 2: Replace any double spaces inside the text
    # 'Nero  Panel warm' becomes 'Nero Panel warm'
    df_clean[col] = df_clean[col].str.replace('  ', ' ', regex=False)

    # Step 3: Apply Title Case
    # 'WETCO WARM' becomes 'Wetco Warm'
    # 'panel light' becomes 'Panel Light'
    # 'nero panel warm' becomes 'Nero Panel Warm'
    df_clean[col] = df_clean[col].str.title()

# Also strip spaces from Specification — just spaces, no case change
df_clean['Specification'] = df_clean['Specification'].str.strip()

print("Text standardization complete.")
print()

# Verify by showing the specific products we knew had problems
problem_ids = ['P0009', 'P0017', 'P0020']
print("Checking the known problem rows:")
print()
print(df_clean[df_clean['Product_ID'].isin(problem_ids)]
      [['Product_ID', 'Product_Name', 'Category']].to_string(index=False))


Text standardization complete.

Checking the known problem rows:

Product_ID    Product_Name    Category
     P0009 Nero Panel Warm Panel Light
     P0017  Tesla Panel Bk Panel Light
     P0020      Wetco Warm Panel Light


In [14]:
# First, let's see the problem clearly
# Show all products currently in both Globe-related categories
globe_mask = df_clean['Category'].isin(['Plastic Globe', 'Globe'])
globe_products = df_clean[globe_mask][['Product_ID', 'Product_Name',
                                        'Category', 'Specification']]

print("Products currently split across two category names:")
print()
print(globe_products.to_string(index=False))
print()
print("'Plastic Globe' count:", (df_clean['Category'] == 'Plastic Globe').sum())
print("'Globe' count        :", (df_clean['Category'] == 'Globe').sum())


Products currently split across two category names:

Product_ID         Product_Name      Category Specification
     P0128        Plastic Globe Plastic Globe      300 Size
     P0129 Plastic Globe Jemaro Plastic Globe      300 Size
     P0130        Plastic Globe Plastic Globe      360 Size
     P0131        Plastic Globe Plastic Globe      400 Size
     P0132 Plastic Globe Jemaro Plastic Globe      400 Size
     P0133        Plastic Globe Plastic Globe      500 Size
     P0145        Plastic Globe         Globe      300 Size
     P0146        Plastic Globe         Globe      360 Size
     P0147 Plastic Globe Jemaro         Globe      360 Size
     P0148        Plastic Globe         Globe      400 Size
     P0149 Plastic Globe Jemaro         Globe      400 Size
     P0150        Plastic Globe         Globe      500 Size

'Plastic Globe' count: 6
'Globe' count        : 6


In [15]:
# Merge: rename 'Globe' to 'Plastic Globe'
# We keep 'Plastic Globe' as the standard name because it is more descriptive
df_clean['Category'] = df_clean['Category'].replace('Globe', 'Plastic Globe')

# Verify
print("After fix:")
print("'Plastic Globe' count:", (df_clean['Category'] == 'Plastic Globe').sum())
print("'Globe' count        :", (df_clean['Category'] == 'Globe').sum())
print()

# Show all unique categories now
print("All categories after fix:")
for cat in sorted(df_clean['Category'].unique()):
    count = (df_clean['Category'] == cat).sum()
    print(f"  {cat:<22} {count} products")


After fix:
'Plastic Globe' count: 12
'Globe' count        : 0

All categories after fix:
  Breaker                44 products
  Grand                  10 products
  Haud                   13 products
  Lamp                   31 products
  Magnetic Light         11 products
  Mirror Light           4 products
  Panel Light            20 products
  Pawza                  7 products
  Plastic Globe          12 products
  Spot Light             25 products
  Strip Light            12 products
  Vegas                  12 products


In [16]:
# Dictionary of corrections
# Format: 'wrong spelling in data': 'correct spelling'
name_corrections = {
    'Contractor':           'Contactor',
    'Cheng Over':           'Change Over',
    'Haveles 1 Phase':      'Havells 1 Phase',
    'Delix Contractor':     'Delix Contactor',
    'Shinder Contractor':   'Shinder Contactor',
    'Fuse Holder Andel':    'Fuse Holder Andeli',
    'Chargable Lamp':       'Chargeable Lamp',
}

# Apply all corrections at once
df_clean['Product_Name'] = df_clean['Product_Name'].replace(name_corrections)

print("Typo corrections applied.")
print()

# Verify each correction actually worked
print("Verification — checking corrected names:")
print()
for wrong, correct in name_corrections.items():
    # Count how many rows now have the correct name
    correct_count = (df_clean['Product_Name'] == correct).sum()
    # Count how many rows still have the wrong name (should be 0)
    wrong_count   = (df_clean['Product_Name'] == wrong).sum()
    print(f"  '{correct:<25}' found: {correct_count}   |   '{wrong}' remaining: {wrong_count}")


Typo corrections applied.

Verification — checking corrected names:

  'Contactor                ' found: 4   |   'Contractor' remaining: 0
  'Change Over              ' found: 2   |   'Cheng Over' remaining: 0
  'Havells 1 Phase          ' found: 2   |   'Haveles 1 Phase' remaining: 0
  'Delix Contactor          ' found: 2   |   'Delix Contractor' remaining: 0
  'Shinder Contactor        ' found: 1   |   'Shinder Contractor' remaining: 0
  'Fuse Holder Andeli       ' found: 2   |   'Fuse Holder Andel' remaining: 0
  'Chargeable Lamp          ' found: 1   |   'Chargable Lamp' remaining: 0


In [17]:
# First, show the products we are about to move
# so we can confirm we are targeting the right rows
products_to_move = df_clean[df_clean['Product_ID'].isin(['P0137', 'P0138'])]

print("Products before re-categorization:")
print()
print(products_to_move[['Product_ID', 'Product_Name',
                          'Category', 'Purchase_Price_ETB']].to_string(index=False))


Products before re-categorization:

Product_ID Product_Name       Category  Purchase_Price_ETB
     P0137    Phase Bar Magnetic Light                1500
     P0138 Power Supply Magnetic Light                2500


In [18]:
# Re-assign the category for these two specific products
# We target by Product_ID so we only change exactly what we intend to change
df_clean.loc[df_clean['Product_ID'].isin(['P0137', 'P0138']), 'Category'] = 'Electrical Component'

# Verify the change
products_after = df_clean[df_clean['Product_ID'].isin(['P0137', 'P0138'])]

print("Products after re-categorization:")
print()
print(products_after[['Product_ID', 'Product_Name',
                        'Category', 'Purchase_Price_ETB']].to_string(index=False))

print()

# Show updated category list
print("Updated category counts:")
print()
for cat in sorted(df_clean['Category'].unique()):
    count = (df_clean['Category'] == cat).sum()
    print(f"  {cat:<22} {count} products")


Products after re-categorization:

Product_ID Product_Name             Category  Purchase_Price_ETB
     P0137    Phase Bar Electrical Component                1500
     P0138 Power Supply Electrical Component                2500

Updated category counts:

  Breaker                44 products
  Electrical Component   2 products
  Grand                  10 products
  Haud                   13 products
  Lamp                   31 products
  Magnetic Light         9 products
  Mirror Light           4 products
  Panel Light            20 products
  Pawza                  7 products
  Plastic Globe          12 products
  Spot Light             25 products
  Strip Light            12 products
  Vegas                  12 products


In [19]:
# Fix P0107 — Chint 1 Phase 10A
# Confirmed correct price: 300 ETB (was incorrectly entered as 2500)
df_clean.loc[df_clean['Product_ID'] == 'P0107', 'Purchase_Price_ETB'] = 300

# Fix P0123 — Fuse Holder Andeli
# Confirmed correct price: 500 ETB (was incorrectly entered as 2500)
df_clean.loc[df_clean['Product_ID'] == 'P0123', 'Purchase_Price_ETB'] = 500

print("Prices corrected.")
print()

# Add the Needs_Review column — all False since we have no unresolved issues
df_clean['Needs_Review'] = False

print("Needs_Review column added. All products verified.")
print()

# Verify the corrections
corrected = df_clean[df_clean['Product_ID'].isin(['P0107', 'P0123'])]
print("Verified prices:")
print()
print(corrected[['Product_ID', 'Product_Name',
                  'Specification', 'Purchase_Price_ETB',
                  'Needs_Review']].to_string(index=False))


Prices corrected.

Needs_Review column added. All products verified.

Verified prices:

Product_ID       Product_Name Specification  Purchase_Price_ETB  Needs_Review
     P0107      Chint 1 Phase           10A                 300         False
     P0123 Fuse Holder Andeli           N/A                 500         False


In [20]:
# Calculate total stock value per product
# Formula: how many units we have × what each unit cost us to buy
df_clean['Total_Stock_Value_ETB'] = df_clean['Quantity'] * df_clean['Purchase_Price_ETB']

print("Total_Stock_Value_ETB column added.")
print()

# Show the grand total — the total capital invested in all inventory
total_value = df_clean['Total_Stock_Value_ETB'].sum()
print(f"Total Inventory Value: {total_value:,.0f} ETB")
print()

# Show top 10 products by stock value
# These are the products with the most capital tied up in them
print("Top 10 products by stock value:")
print()

top_10 = (
    df_clean[['Product_Name', 'Category', 'Quantity',
              'Purchase_Price_ETB', 'Total_Stock_Value_ETB']]
    .sort_values('Total_Stock_Value_ETB', ascending=False)
    .head(10)
)

print(top_10.to_string(index=False))



Total_Stock_Value_ETB column added.

Total Inventory Value: 944,560 ETB

Top 10 products by stock value:

      Product_Name      Category  Quantity  Purchase_Price_ETB  Total_Stock_Value_ETB
     Plastic Globe Plastic Globe        17                2200                  37400
     Plastic Globe Plastic Globe        17                2200                  37400
Triple White Strip   Strip Light       200                 145                  29000
      Cristal Spot    Spot Light        61                 420                  25620
        Kk Spot Bk    Spot Light       108                 230                  24840
  Tesla Spot White    Spot Light       100                 230                  23000
     Plastic Globe Plastic Globe        14                1500                  21000
     Plastic Globe Plastic Globe        14                1500                  21000
     Himel 3 Phase       Breaker         4                4800                  19200
   Nero Panel Warm   Panel Light  

In [21]:
# Show all Plastic Globe products side by side
globe_all = df_clean[df_clean['Category'] == 'Plastic Globe'][
    ['Product_ID', 'Product_Name', 'Specification',
     'Quantity', 'Purchase_Price_ETB']
].sort_values('Specification')

print("All Plastic Globe products:")
print()
print(globe_all.to_string(index=False))


All Plastic Globe products:

Product_ID         Product_Name Specification  Quantity  Purchase_Price_ETB
     P0128        Plastic Globe      300 Size        14                1500
     P0129 Plastic Globe Jemaro      300 Size         2                 700
     P0145        Plastic Globe      300 Size        14                1500
     P0130        Plastic Globe      360 Size         1                1500
     P0146        Plastic Globe      360 Size         1                1500
     P0147 Plastic Globe Jemaro      360 Size         2                 700
     P0131        Plastic Globe      400 Size        17                2200
     P0132 Plastic Globe Jemaro      400 Size         3                 900
     P0148        Plastic Globe      400 Size        17                2200
     P0149 Plastic Globe Jemaro      400 Size         3                 900
     P0133        Plastic Globe      500 Size         2                2800
     P0150        Plastic Globe      500 Size         2    

In [22]:
# Define the duplicate Product IDs to remove
duplicates_to_remove = ['P0145', 'P0146', 'P0147', 'P0148', 'P0149', 'P0150']

# Show what we are about to delete — always confirm before deleting
print("Rows to be deleted:")
print()
print(df_clean[df_clean['Product_ID'].isin(duplicates_to_remove)]
      [['Product_ID', 'Product_Name', 'Specification',
        'Quantity', 'Purchase_Price_ETB']].to_string(index=False))

print()
print(f"Row count before deletion: {len(df_clean)}")


Rows to be deleted:

Product_ID         Product_Name Specification  Quantity  Purchase_Price_ETB
     P0145        Plastic Globe      300 Size        14                1500
     P0146        Plastic Globe      360 Size         1                1500
     P0147 Plastic Globe Jemaro      360 Size         2                 700
     P0148        Plastic Globe      400 Size        17                2200
     P0149 Plastic Globe Jemaro      400 Size         3                 900
     P0150        Plastic Globe      500 Size         2                2800

Row count before deletion: 201


In [23]:
# Remove the duplicate rows
# ~ means NOT — so this keeps all rows where Product_ID is NOT in the list
df_clean = df_clean[~df_clean['Product_ID'].isin(duplicates_to_remove)]

# Reset the index after deletion
# When rows are deleted, the index numbers have gaps (e.g. 0,1,2...144,148...)
# reset_index() renumbers them cleanly from 0 to 194
# drop=True means: don't add the old index as a new column
df_clean = df_clean.reset_index(drop=True)

print(f"Row count after deletion : {len(df_clean)}")
print()

# Verify Plastic Globe now shows 6 products
globe_count = (df_clean['Category'] == 'Plastic Globe').sum()
print(f"Plastic Globe products remaining: {globe_count}")
print()

# Show the remaining Plastic Globe products
print("Plastic Globe products after fix:")
print()
print(df_clean[df_clean['Category'] == 'Plastic Globe']
      [['Product_ID', 'Product_Name',
        'Specification', 'Quantity',
        'Purchase_Price_ETB']].to_string(index=False))


Row count after deletion : 195

Plastic Globe products remaining: 6

Plastic Globe products after fix:

Product_ID         Product_Name Specification  Quantity  Purchase_Price_ETB
     P0128        Plastic Globe      300 Size        14                1500
     P0129 Plastic Globe Jemaro      300 Size         2                 700
     P0130        Plastic Globe      360 Size         1                1500
     P0131        Plastic Globe      400 Size        17                2200
     P0132 Plastic Globe Jemaro      400 Size         3                 900
     P0133        Plastic Globe      500 Size         2                2800


In [24]:
# Recalculate with correct data — 195 rows, corrected prices
df_clean['Total_Stock_Value_ETB'] = df_clean['Quantity'] * df_clean['Purchase_Price_ETB']

# Show the corrected total
total_value = df_clean['Total_Stock_Value_ETB'].sum()
print(f"Corrected Total Inventory Value: {total_value:,.0f} ETB")
print()

# Show the corrected top 10
print("Top 10 products by stock value (corrected):")
print()

top_10 = (
    df_clean[['Product_Name', 'Category', 'Quantity',
              'Purchase_Price_ETB', 'Total_Stock_Value_ETB']]
    .sort_values('Total_Stock_Value_ETB', ascending=False)
    .head(10)
)

print(top_10.to_string(index=False))


Corrected Total Inventory Value: 874,960 ETB

Top 10 products by stock value (corrected):

      Product_Name      Category  Quantity  Purchase_Price_ETB  Total_Stock_Value_ETB
     Plastic Globe Plastic Globe        17                2200                  37400
Triple White Strip   Strip Light       200                 145                  29000
      Cristal Spot    Spot Light        61                 420                  25620
        Kk Spot Bk    Spot Light       108                 230                  24840
  Tesla Spot White    Spot Light       100                 230                  23000
     Plastic Globe Plastic Globe        14                1500                  21000
     Himel 3 Phase       Breaker         4                4800                  19200
   Nero Panel Warm   Panel Light        20                 850                  17000
         Rgb Strip   Strip Light       100                 165                  16500
      Bright Panel   Panel Light        23       

In [25]:
print('=' * 50)
print('     FINAL QUALITY REPORT — Nate Data')
print('=' * 50)
print()

# 1. Shape
print(f'Total products    : {len(df_clean)}')
print(f'Total columns     : {len(df_clean.columns)}')
print()

# 2. Column names and data types
print('Column names and data types:')
for col, dtype in df_clean.dtypes.items():
    print(f'  {col:<25} {str(dtype)}')
print()

# 3. Missing values
print('Missing values per column:')
missing = df_clean.isnull().sum()
if missing.sum() == 0:
    print('  None. All columns are complete.')
else:
    print(missing[missing > 0])
print()

# 4. Category summary
print('Categories and product counts:')
for cat in sorted(df_clean['Category'].unique()):
    count = (df_clean['Category'] == cat).sum()
    print(f'  {cat:<22} {count} products')
print()

# 5. Key business numbers
total_qty   = df_clean['Quantity'].sum()
total_value = df_clean['Total_Stock_Value_ETB'].sum()
total_cats  = df_clean['Category'].nunique()
out_of_stock = (df_clean['Quantity'] == 0).sum()

print('Key business numbers:')
print(f'  Total inventory value : {total_value:,.0f} ETB')
print(f'  Total units in stock  : {total_qty:,}')
print(f'  Total categories      : {total_cats}')
print(f'  Out of stock products : {out_of_stock}')


     FINAL QUALITY REPORT — Nate Data

Total products    : 195
Total columns     : 10

Column names and data types:
  Product_ID                str
  Product_Name              str
  Category                  str
  Specification             str
  Quantity                  int64
  Purchase_Price_ETB        int64
  Location                  str
  Count_Date                datetime64[s]
  Needs_Review              bool
  Total_Stock_Value_ETB     int64

Missing values per column:
  None. All columns are complete.

Categories and product counts:
  Breaker                44 products
  Electrical Component   2 products
  Grand                  10 products


  Haud                   13 products
  Lamp                   31 products
  Magnetic Light         9 products
  Mirror Light           4 products
  Panel Light            20 products
  Pawza                  7 products
  Plastic Globe          6 products
  Spot Light             25 products
  Strip Light            12 products
  Vegas                  12 products

Key business numbers:
  Total inventory value : 874,960 ETB
  Total units in stock  : 2,511
  Total categories      : 13
  Out of stock products : 12


In [26]:
# Save as Excel — for human review and sharing
df_clean.to_excel(cleaned_xlsx, index=False)
print(f'Saved Excel : {cleaned_xlsx}')

# Save as CSV — for all future Python work
# encoding='utf-8-sig' ensures the file opens correctly in Excel
# and handles any special characters properly
df_clean.to_csv(cleaned_csv, index=False, encoding='utf-8-sig')
print(f'Saved CSV   : {cleaned_csv}')

print()
print('Phase 2 — Data Cleaning: COMPLETE')
print()
print(f'  Raw data      : 201 rows, 8 columns')
print(f'  Cleaned data  : {len(df_clean)} rows, {len(df_clean.columns)} columns')
print(f'  Rows removed  : {201 - len(df_clean)} (6 duplicate Plastic Globe entries)')
print(f'  Columns added : 2 (Needs_Review, Total_Stock_Value_ETB)')
print(f'  Total value   : 874,960 ETB')
print()
print('Ready for Phase 3: Exploratory Data Analysis.')


Saved Excel : c:\Users\DELL\Documents\Summer_Projects\data\cleaned\inventory_cleaned.xlsx
Saved CSV   : c:\Users\DELL\Documents\Summer_Projects\data\cleaned\inventory_cleaned.csv

Phase 2 — Data Cleaning: COMPLETE

  Raw data      : 201 rows, 8 columns
  Cleaned data  : 195 rows, 10 columns
  Rows removed  : 6 (6 duplicate Plastic Globe entries)
  Columns added : 2 (Needs_Review, Total_Stock_Value_ETB)
  Total value   : 874,960 ETB

Ready for Phase 3: Exploratory Data Analysis.
